## 1. Import tools

In [ ]:

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS #Builds a vector index and searches for similar text.
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate


## 2. Load Documents

In [4]:
loader = WebBaseLoader(
    web_paths=(
        "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    )
)

documents = loader.load()

#### extra

In [5]:
documents[0].page_content

'\n\n\n\nRetrieval-augmented generation - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nRAG and LLM limitations\n\n\n\n\n\n\n\n\n2\nProcess\n\n\n\n\nToggle Process subsection\n\n\n\n\n\n2.1\nRAG key stages\n\n\n\n\n\n\n\n\n\n\n3\nApplications\n\n\n\n\n\n\n\n\n4\nImprovements\n\n\n\n

In [6]:
documents[0].metadata  

{'source': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation',
 'title': 'Retrieval-augmented generation - Wikipedia',
 'language': 'en'}

## 3. Split the documents into chunk
s

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

#### extrA

In [9]:
print("Total chunks:", len(chunks))

Total chunks: 43


In [10]:
print(chunks[0].page_content)

Retrieval-augmented generation - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create account


Log in





























Contents
move to sidebar
hide




(Top)





1
RAG and LLM limitations








2
Process




Toggle Process subsection





2.1
RAG key stages










3
Applications








4
Improvements




Toggle Improvements subsection





4.1
Encoder


In [11]:
print(chunks[0].metadata)

{'source': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'title': 'Retrieval-augmented generation - Wikipedia', 'language': 'en'}


## 4. Convert text into embeddings

In [12]:

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)


#### EXTRA

In [13]:
vector = embeddings.embed_query(
    "What is retrieval-augmented generation?"
)

print(type(vector))
print("Vector dimensions:", len(vector))

<class 'list'>
Vector dimensions: 768


In [14]:
vector

[-0.028587947,
 0.07043654,
 -0.16599219,
 -0.108682916,
 0.03661166,
 -0.0014814341,
 0.006271537,
 0.039418913,
 -0.023516342,
 0.020854143,
 0.0290757,
 0.01719987,
 0.11288878,
 -0.004727543,
 -0.06288972,
 -0.0304338,
 -0.0046281465,
 -0.0056907968,
 -0.065232985,
 0.020512765,
 0.043290023,
 -0.036522385,
 -0.026000697,
 0.051743902,
 0.065870695,
 -0.016242463,
 -0.012434331,
 0.0063477396,
 -0.026516285,
 0.009057881,
 0.060804218,
 0.0050345627,
 -0.014456662,
 0.067577384,
 -0.02684041,
 0.0016204736,
 0.032808956,
 0.028969936,
 -0.0057975473,
 0.018547596,
 -0.02471766,
 -0.012912003,
 0.037469868,
 -0.01983171,
 0.0023717622,
 0.023726795,
 0.03481019,
 0.015523619,
 0.052284133,
 -0.04309048,
 0.01630095,
 -0.009257126,
 -0.03269989,
 -0.011603314,
 0.09888498,
 0.008087499,
 -0.047406174,
 0.024887338,
 -0.023098046,
 -0.049829748,
 0.066727586,
 0.09322006,
 -0.026713056,
 0.08210363,
 0.057736024,
 -0.049408387,
 -0.0006287644,
 0.022982802,
 -0.017525252,
 -0.01681158

## 5. Store the vectors in FAISS

In [16]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

#### EXTRA

In [17]:
print("Indexed chunks:", vectorstore.index.ntotal)

Indexed chunks: 43


#### TO SAVE THE MODEL(FAISS)

In [18]:
# Save the index
vectorstore.save_local("rag_faiss_index")

# Later, reload it
vectorstore = FAISS.load_local(
    "rag_faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

## 6.Create the Retriever

In [19]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)


#### Extra(to use retriever)

In [20]:
question = "How does RAG reduce hallucinations?"

retrieved_docs = retriever.invoke(question)

for doc in retrieved_docs:
    print(doc.page_content)
    print(doc.metadata)

RAG improves LLMs by incorporating information retrieval before generating responses.[4] Unlike LLMs that rely on static training data, RAG pulls relevant text from databases, uploaded documents, or web sources.[1] According to Ars Technica, "RAG is a way of improving LLM performance, in essence by blending the LLM process with a web search or other document look-up process to help LLMs stick to the facts." This method helps reduce AI hallucinations,[4] which have caused chatbots to describe policies that don't exist, or recommend nonexistent legal cases to lawyers that are looking for citations to support their arguments.[5]
{'source': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'title': 'Retrieval-augmented generation - Wikipedia', 'language': 'en'}
While RAG improves the accuracy of large language models (LLMs), it does not eliminate all challenges. One limitation is that while RAG reduces the need for frequent model retraining, it does not remove it entirely. Ad

## 7.Initialize the Ollama language model

In [21]:

llm = ChatOllama(
    model="qwen3:4b",
    temperature=0
)


## 8 — Create the prompt template

In [22]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.
If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{question}
""")

## 9 — Retrieve the evidence for a question

In [23]:

question = "What is retrieval-augmented generation?"

retrieved_docs = retriever.invoke(question)


#### extra(retrieved_docs)

In [24]:
retrieved_docs

[Document(id='b797ff68-bc11-48d1-b0af-73f56dbeaac9', metadata={'source': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation', 'title': 'Retrieval-augmented generation - Wikipedia', 'language': 'en'}, page_content="Type of information retrieval using LLMs\nRetrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources.[1][2] With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data.[3] This allows LLMs to use domain-specific and/or updated information that is not available in the training data.[3] For example, this enables LLM-based chatbots to access internal company data or generate responses based on authoritative sources. The technique was first proposed in 2020 and has since become a widely adopted approach in modern AI systems."),
 Document(id='9a7bf919-af4b-

## 10 — Combine the retrieved chunks into context

In [25]:
context = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)


## 11 — Insert the question and context into the prompt

In [27]:
messages = prompt.invoke({
    "context": context,
    "question": question
})


## 12 — Generate the final answer

In [29]:

response = llm.invoke(messages)

print(response.content)


Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data, allowing LLMs to use domain-specific and/or updated information that is not available in the training data.
